# 03 – HMM + Viterbi route reconstruction
Hidden states = candidate arcs of each GPS point. Emission = log of the MLP probability.
Transition = road connectivity + distance consistency (route distance along the network vs
GPS distance), U-turn penalty, jump penalty for unconnected arcs; uniform after a time gap.
Viterbi finds the most likely connected sequence. Compared with classifier-only (argmax)
on the 10 student_test trips. Decoder parameters chosen on student_test only (never evaluator).
Uses the same code as the API: src/pipeline.py, src/scoring.py, src/hmm.py.

In [ ]:
import os, sys, shutil, importlib
from google.colab import drive
drive.mount("/content/drive", force_remount=True)        # pick up newly uploaded files
OUT = "/content/drive/MyDrive/pathsnap"
print("py files on Drive:", sorted(f for f in os.listdir(OUT) if f.endswith(".py")))

# Copy them to a local folder and import from there (avoids Drive caching issues)
os.makedirs("/content/src", exist_ok=True)
for f in ["pipeline.py", "scoring.py", "hmm.py"]:
    shutil.copy(os.path.join(OUT, f), "/content/src/")
sys.path.insert(0, "/content/src")
importlib.invalidate_caches()
import pipeline, scoring, hmm
print("imported OK")

Mounted at /content/drive
py files on Drive: ['hmm.py', 'main.py', 'pipeline.py', 'scoring.py']
imported OK


In [ ]:
import time, zipfile, joblib
import numpy as np, pandas as pd

with zipfile.ZipFile(f"{OUT}/Student folder.zip") as z:
    z.extractall("/content/data")
TEST_ROOT = "/content/data/Student folder/student_test"
test_trips = sorted(os.listdir(TEST_ROOT))

model = joblib.load(f"{OUT}/models/MLP_ctx.joblib")      # swap to MLP_ctx_tuned.joblib later
print(len(test_trips), "student_test trips | features:", len(pipeline.FEATURES))

10 student_test trips | features: 15


In [ ]:
cache = {}
t0 = time.time()
for t in test_trips:
    d = os.path.join(TEST_ROOT, t)
    track, nodes, arcs = pipeline.load_trip_inputs(d)
    track, nodes, arcs, cands = pipeline.make_features(track, nodes, arcs)
    cands["p"] = model.predict_proba(cands[pipeline.FEATURES])[:, 1]
    cache[t] = {"dir": d, "track": track, "nodes": nodes, "arcs": arcs, "cands": cands}
print(f"features + probabilities for {len(cache)} trips in {time.time() - t0:.0f} s")

def evaluate(decoder_params=None, use_hmm=True):
    """Decode every student_test trip and score it. Returns per-trip DataFrame."""
    rows = []
    for t, c in cache.items():
        t1 = time.time()
        if use_hmm:
            arc, frac, prob = hmm.viterbi_decoder(c["cands"], c["track"], c["arcs"], decoder_params)
        else:
            arc, frac, prob = pipeline.decode_argmax(c["cands"], len(c["track"]))
        dec_time = time.time() - t1
        pred = {"track": c["track"], "arcs": c["arcs"], "pred_arc": arc}
        m = scoring.score_trip(pred, c["dir"])
        m["route_continuity"] = pipeline.route_continuity(arc, c["track"].segment.values, c["arcs"])
        m.update({"trip": t, "decode_time_s": round(dec_time, 2)})
        rows.append(m)
    return pd.DataFrame(rows)

def summary(df):
    w = df.n_points_scored
    return {"top1_acc": round((df.top1_accuracy * w).sum() / w.sum(), 4),     # point-weighted overall
            "continuity": round(df.route_continuity.mean(), 4),
            "decode_time_s": round(df.decode_time_s.sum(), 1)}

features + probabilities for 10 trips in 5 s


In [ ]:
base = evaluate(use_hmm=False)
hmm_res = evaluate(use_hmm=True)
print("Classifier only (argmax):", summary(base))
print("Classifier + HMM/Viterbi:", summary(hmm_res))

per_trip = base[["trip", "n_points", "top1_accuracy", "route_continuity"]].merge(
    hmm_res[["trip", "top1_accuracy", "route_continuity"]], on="trip", suffixes=("_argmax", "_hmm"))
per_trip["gain"] = per_trip.top1_accuracy_hmm - per_trip.top1_accuracy_argmax
per_trip.to_csv(f"{OUT}/hmm_vs_argmax_per_trip.csv", index=False)
per_trip.round(4)

Classifier only (argmax): {'top1_acc': np.float64(0.8922), 'continuity': np.float64(0.9581), 'decode_time_s': np.float64(0.0)}
Classifier + HMM/Viterbi: {'top1_acc': np.float64(0.9214), 'continuity': np.float64(0.9746), 'decode_time_s': np.float64(7.5)}


,trip,n_points,top1_accuracy_argmax,route_continuity_argmax,top1_accuracy_hmm,route_continuity_hmm,gain
0,00000000,2503,0.9521,0.9848,0.9557,0.9944,0.0036
1,00000007,1354,0.9505,0.9993,0.9948,0.9993,0.0443
2,00000009,1049,0.8837,0.9237,0.8942,0.9447,0.0105
3,00000040,2995,0.9499,0.9793,0.9629,0.9913,0.0130
4,00000052,262,0.9733,0.9502,0.9924,0.9540,0.0191
5,00000056,3958,0.9183,0.9297,0.9560,0.9626,0.0376
6,00000065,768,0.8659,0.9713,0.8763,0.9830,0.0104
7,00000076,1486,0.9354,0.9327,0.9603,0.9684,0.0249
8,00000077,3911,0.7435,0.9703,0.8008,0.9859,0.0573
9,00000091,874,0.9153,0.9393,0.9371,0.9622,0.0217


In [ ]:
grid = []
for beta in [10, 20, 40]:
    for jump in [-10, -20]:
        for uturn in [-4, -8]:
            p = {"beta": beta, "jump_logp": jump, "uturn_logp": uturn}
            s = summary(evaluate(p)); s.update(p); grid.append(s)
            print(s)
grid = pd.DataFrame(grid).sort_values("top1_acc", ascending=False)
grid.to_csv(f"{OUT}/hmm_param_grid.csv", index=False)
grid.head()

{'top1_acc': np.float64(0.9172), 'continuity': np.float64(0.9755), 'decode_time_s': np.float64(8.5), 'beta': 10, 'jump_logp': -10, 'uturn_logp': -4}
{'top1_acc': np.float64(0.9248), 'continuity': np.float64(0.9751), 'decode_time_s': np.float64(6.4), 'beta': 10, 'jump_logp': -10, 'uturn_logp': -8}
{'top1_acc': np.float64(0.9189), 'continuity': np.float64(0.9752), 'decode_time_s': np.float64(7.8), 'beta': 10, 'jump_logp': -20, 'uturn_logp': -4}
{'top1_acc': np.float64(0.9256), 'continuity': np.float64(0.9748), 'decode_time_s': np.float64(6.2), 'beta': 10, 'jump_logp': -20, 'uturn_logp': -8}
{'top1_acc': np.float64(0.9143), 'continuity': np.float64(0.9752), 'decode_time_s': np.float64(8.5), 'beta': 20, 'jump_logp': -10, 'uturn_logp': -4}
{'top1_acc': np.float64(0.9202), 'continuity': np.float64(0.9749), 'decode_time_s': np.float64(7.8), 'beta': 20, 'jump_logp': -10, 'uturn_logp': -8}
{'top1_acc': np.float64(0.9158), 'continuity': np.float64(0.9749), 'decode_time_s': np.float64(6.9), 'beta

,top1_acc,continuity,decode_time_s,beta,jump_logp,uturn_logp
3,0.9256,0.9748,6.2,10,-20,-8
1,0.9248,0.9751,6.4,10,-10,-8
7,0.9213,0.9745,8.2,20,-20,-8
5,0.9202,0.9749,7.8,20,-10,-8
2,0.9189,0.9752,7.8,10,-20,-4
